In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_score, recall_score, f1_score
)
import timm
import cv2
from NoduleDS import NoduleDataset

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 916   # change to 284 or 916 for other runs
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [33]:
# ── Configuration ─────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
NUM_EPOCHS    = 10
LEARNING_RATE = 3e-4
IMAGE_SIZE    = 224
NUM_WORKERS   = 4
WEIGHT_DECAY  = 0.01
ATTENTION_WEIGHT = 1.0
SIGMA_SCALE   = 1.0

MODEL_NAME    = f'spatial_attention_only_seed{SEED}.pth'

# Path to the fine-tuned ResNet-50 baseline checkpoint
BASELINE_CHECKPOINT = './rnnet_916.pth'

split_base_path  = './dataset_nodule21/cxr_images/proccessed_data/split_data'
train_images_path = f'{split_base_path}/train/images'
val_images_path   = f'{split_base_path}/val/images'
test_images_path  = f'{split_base_path}/test/images'

# No-aug CSV for training (required by spatial attention due to bbox alignment)
train_csv_path = f'{split_base_path}/train/metadata_no_aug.csv'
val_csv_path   = f'{split_base_path}/val/metadata_val.csv'
test_csv_path  = f'{split_base_path}/test/metadata_test.csv'

In [34]:
# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ── Datasets & Loaders ────────────────────────────────────────────────────────
print('Loading datasets...')
train_df = pd.read_csv(train_csv_path)
val_df   = pd.read_csv(val_csv_path)
test_df  = pd.read_csv(test_csv_path)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# return_bbox=True — spatial attention loss needs bbox coordinates
train_dataset = NoduleDataset(train_df, train_images_path, transform=train_transform, return_bbox=True)
val_dataset   = NoduleDataset(val_df,   val_images_path,   transform=val_transform,   return_bbox=True)
test_dataset  = NoduleDataset(test_df,  test_images_path,  transform=val_transform,   return_bbox=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

Loading datasets...
Train: 3657 | Val: 782 | Test: 785


In [35]:
# ── Spatial Attention Module (same as resnet-50-guidedv3) ─────────────────────
class SpatialAttentionModule(nn.Module):
    def __init__(self, in_channels=2048, reduction=8):
        super().__init__()
        self.conv1    = nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1)
        self.bn1      = nn.BatchNorm2d(in_channels // reduction)
        self.relu     = nn.ReLU(inplace=True)
        self.conv2    = nn.Conv2d(in_channels // reduction, in_channels // reduction, kernel_size=3, padding=1)
        self.bn2      = nn.BatchNorm2d(in_channels // reduction)
        self.conv_out = nn.Conv2d(in_channels // reduction, 1, kernel_size=1)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        att = self.relu(self.bn1(self.conv1(x)))
        att = self.relu(self.bn2(self.conv2(att)))
        attention = self.sigmoid(self.conv_out(att))
        return attention, x * attention


# ── Full Model: RNNet-MST backbone + Spatial Attention only trainable ─────────
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=2, mlp_ratio=1.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, dim), nn.Dropout(dropout)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        if H * W > 196:
            x_down = F.adaptive_avg_pool2d(x, (14, 14))
            H_d, W_d = 14, 14
            x_seq = x_down.flatten(2).transpose(1, 2)
        else:
            x_seq = x.flatten(2).transpose(1, 2)
            H_d, W_d = H, W
        x_norm = self.norm1(x_seq)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x_seq = x_seq + attn_out
        x_seq = x_seq + self.mlp(self.norm2(x_seq))
        x_out = x_seq.transpose(1, 2).reshape(B, C, H_d, W_d)
        if H_d != H or W_d != W:
            x_out = F.interpolate(x_out, size=(H, W), mode='bilinear', align_corners=False)
        return x + x_out


class RNNetMST_SpatialAttentionOnly(nn.Module):
    """
    Ablation variant: ResNet-50 + all 4 MST stages (frozen) + Spatial Attention (trainable)
    Only the spatial attention module and classifier are trained.
    """
    def __init__(self, num_classes=2, num_heads=2, dropout=0.1):
        super().__init__()

        # ── ResNet-50 backbone (will be frozen) ───────────────────────────────
        backbone = timm.create_model('resnet50', pretrained=False, num_classes=0)
        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.act1, backbone.maxpool)
        self.stage1 = backbone.layer1
        self.stage2 = backbone.layer2
        self.stage3 = backbone.layer3
        self.stage4 = backbone.layer4

        # ── MST blocks at all 4 stages (will be frozen) ───────────────────────
        self.trans1 = TransformerBlock(256,  num_heads=2,        mlp_ratio=1.0, dropout=dropout)
        self.trans2 = TransformerBlock(512,  num_heads=2,        mlp_ratio=1.0, dropout=dropout)
        self.trans3 = TransformerBlock(1024, num_heads=num_heads, mlp_ratio=1.0, dropout=dropout)
        self.trans4 = TransformerBlock(2048, num_heads=num_heads, mlp_ratio=1.0, dropout=dropout)

        # ── Spatial Attention (TRAINABLE) ─────────────────────────────────────
        self.spatial_attention = SpatialAttentionModule(in_channels=2048)

        # ── Classifier (TRAINABLE) ────────────────────────────────────────────
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Linear(2048, num_classes)

    def forward(self, x, return_attention=False):
        x = self.stem(x)
        x = self.trans1(self.stage1(x))
        x = self.trans2(self.stage2(x))
        x = self.trans3(self.stage3(x))
        x = self.trans4(self.stage4(x))   # [B, 2048, 7, 7]

        attention_map, x = self.spatial_attention(x)

        x = self.global_pool(x).flatten(1)
        out = self.classifier(x)

        if return_attention:
            return out, attention_map
        return out

    def get_attention_map(self, x):
        with torch.no_grad():
            x = self.stem(x)
            x = self.trans1(self.stage1(x))
            x = self.trans2(self.stage2(x))
            x = self.trans3(self.stage3(x))
            x = self.trans4(self.stage4(x))
            attention_map, _ = self.spatial_attention(x)
        return attention_map

In [36]:
# ── Load model and pretrained weights ─────────────────────────────────────────
print('Initialising model...')
model = RNNetMST_SpatialAttentionOnly(num_classes=2, num_heads=2, dropout=0.1)

print(f'Loading pretrained RNNet-MST weights from {BASELINE_CHECKPOINT}...')
checkpoint = torch.load(BASELINE_CHECKPOINT, map_location='cpu')
pretrained_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint

# Load weights — strict=False so missing spatial_attention keys are skipped
missing, unexpected = model.load_state_dict(pretrained_dict, strict=False)
print(f'  Missing keys (expected — new modules): {len(missing)}')
print(f'  Unexpected keys: {len(unexpected)}')

model = model.to(device)

# ── Freeze everything EXCEPT spatial_attention and classifier ─────────────────
for name, param in model.named_parameters():
    if 'spatial_attention' in name or 'classifier' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
total     = trainable + frozen
print(f'\nParameter summary:')
print(f'  Trainable : {trainable:,} ({100*trainable/total:.2f}%)')
print(f'  Frozen    : {frozen:,} ({100*frozen/total:.2f}%)')
print(f'  Total     : {total:,}')
print('\n✓ Frozen : ResNet-50 backbone + all MST blocks')
print('✓ Trainable : Spatial Attention module + Classifier')

Initialising model...
Loading pretrained RNNet-MST weights from ./rnnet_916.pth...
  Missing keys (expected — new modules): 14
  Unexpected keys: 0

Parameter summary:
  Trainable : 1,120,003 (1.93%)
  Frozen    : 56,969,792 (98.07%)
  Total     : 58,089,795

✓ Frozen : ResNet-50 backbone + all MST blocks
✓ Trainable : Spatial Attention module + Classifier


In [37]:
# ── Loss, optimiser, scheduler ────────────────────────────────────────────────
train_labels_arr = train_df['label'].values
class_counts  = np.bincount(train_labels_arr)
class_weights = len(train_labels_arr) / (len(class_counts) * class_counts)
class_weights = torch.FloatTensor(class_weights).to(device)
print(f'Class weights — No Nodule: {class_weights[0]:.4f} | Nodule: {class_weights[1]:.4f}')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

Class weights — No Nodule: 0.6971 | Nodule: 1.7684


In [38]:
# ── Attention loss helpers (same as resnet-50-guidedv3) ───────────────────────
def bbox_to_gaussian_mask(bbox, target_size=(7, 7), sigma_scale=1.0):
    batch_size = bbox.shape[0]
    h, w = target_size
    y_coords = torch.linspace(0, 1, h, device=bbox.device).view(-1, 1).expand(h, w)
    x_coords = torch.linspace(0, 1, w, device=bbox.device).view(1, -1).expand(h, w)
    masks = []
    for i in range(batch_size):
        x, y, box_w, box_h = bbox[i]
        cx = x + box_w / 2
        cy = y + box_h / 2
        sigma_x = max((box_w / 2) * sigma_scale, 0.05)
        sigma_y = max((box_h / 2) * sigma_scale, 0.05)
        gaussian = torch.exp(
            -((x_coords - cx) ** 2) / (2 * sigma_x ** 2) -
            ((y_coords - cy) ** 2) / (2 * sigma_y ** 2)
        )
        masks.append(gaussian)
    return torch.stack(masks).unsqueeze(1)


def spatial_attention_loss(attention_map, bbox, label, sigma_scale=1.0):
    positive_mask = label == 1
    if positive_mask.sum() == 0:
        return torch.tensor(0.0, device=attention_map.device)
    pos_attention = attention_map[positive_mask]
    pos_bbox      = bbox[positive_mask]
    target_size   = (pos_attention.shape[2], pos_attention.shape[3])
    gaussian_masks = bbox_to_gaussian_mask(pos_bbox, target_size, sigma_scale)
    mse_loss = F.mse_loss(pos_attention, gaussian_masks)
    att_prob = pos_attention / (pos_attention.sum(dim=(2, 3), keepdim=True) + 1e-8)
    tgt_prob = gaussian_masks / (gaussian_masks.sum(dim=(2, 3), keepdim=True) + 1e-8)
    kl_loss  = F.kl_div((att_prob + 1e-8).log(), tgt_prob, reduction='batchmean')
    return 0.5 * mse_loss + 0.5 * kl_loss

In [39]:
# ── Training and validation functions ─────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, device, attention_weight, sigma_scale):
    model.train()
    running_loss = running_cls = running_att = 0.0
    correct = total = 0
    all_preds, all_labels, all_probs = [], [], []

    pbar = tqdm(loader, desc='Train')
    for images, labels, bboxes in pbar:
        images, labels, bboxes = images.to(device), labels.to(device), bboxes.to(device)
        optimizer.zero_grad()

        outputs, attention_map = model(images, return_attention=True)
        cls_loss = criterion(outputs, labels)
        att_loss = spatial_attention_loss(attention_map, bboxes, labels, sigma_scale)
        loss     = cls_loss + attention_weight * att_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        running_cls  += cls_loss.item()
        running_att  += att_loss.item()

        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().detach().numpy())

        pbar.set_postfix({
            'loss': f'{running_loss/len(pbar):.4f}',
            'cls' : f'{running_cls/len(pbar):.4f}',
            'att' : f'{running_att/len(pbar):.4f}',
            'acc' : f'{100.*correct/total:.2f}%'
        })

    return (
        running_loss / len(loader),
        running_cls  / len(loader),
        running_att  / len(loader),
        100. * correct / total,
        all_preds, all_labels, all_probs
    )


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = total = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels, bboxes in tqdm(loader, desc='Val/Test'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            running_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc  = 100. * correct / total
    precision  = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    recall     = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    f1         = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    auc        = roc_auc_score(all_labels, all_probs)

    return epoch_loss, epoch_acc, all_preds, all_labels, all_probs, precision, recall, f1, auc

In [40]:
# ── Training Loop ─────────────────────────────────────────────────────────────
print('\n' + '='*70)
print('Starting training — Spatial Attention Only (seed={SEED})')
print('='*70)

best_val_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')
    print('-' * 70)

    train_loss, train_cls, train_att, train_acc, train_preds, train_labels, train_probs = train_epoch(
        model, train_loader, criterion, optimizer, device, ATTENTION_WEIGHT, SIGMA_SCALE
    )
    train_auc = roc_auc_score(train_labels, train_probs)

    val_loss, val_acc, val_preds, val_labels, val_probs, val_prec, val_recall, val_f1, val_auc = validate(
        model, val_loader, criterion, device
    )

    scheduler.step()

    print(f'Train — Loss: {train_loss:.4f} (Cls: {train_cls:.4f}, Att: {train_att:.4f}) | Acc: {train_acc:.2f}% | AUC: {train_auc:.4f}')
    print(f'Val   — Loss: {val_loss:.4f} | Acc: {val_acc:.2f}% | Prec: {val_prec:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}')
    print(f'LR: {optimizer.param_groups[0]["lr"]:.6f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch'              : epoch,
            'model_state_dict'   : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc'            : val_acc,
            'val_auc'            : val_auc,
            'val_precision'      : val_prec,
            'val_recall'         : val_recall,
            'val_f1'             : val_f1,
            'seed'               : SEED,
        }, MODEL_NAME)
        print(f'✓ Saved best model (Val F1: {val_f1:.4f} | Val Recall: {val_recall:.4f})')

print('\nTraining complete!')


Starting training — Spatial Attention Only (seed={SEED})

Epoch 1/10
----------------------------------------------------------------------


Val/Test: 100%|██████████| 49/49 [00:13<00:00,  3.56it/s]


Train — Loss: 1.7270 (Cls: 0.4294, Att: 1.2976) | Acc: 77.36% | AUC: 0.8941
Val   — Loss: 0.2270 | Acc: 92.07% | Prec: 0.8052 | Recall: 0.9556 | F1: 0.8740 | AUC: 0.9829
LR: 0.000293
✓ Saved best model (Val F1: 0.8740 | Val Recall: 0.9556)

Epoch 2/10
----------------------------------------------------------------------


Val/Test: 100%|██████████| 49/49 [00:13<00:00,  3.56it/s]


Train — Loss: 1.5828 (Cls: 0.3996, Att: 1.1832) | Acc: 80.12% | AUC: 0.9092
Val   — Loss: 0.2376 | Acc: 91.82% | Prec: 0.8061 | Recall: 0.9422 | F1: 0.8689 | AUC: 0.9762
LR: 0.000271

Epoch 3/10
----------------------------------------------------------------------


...

Train — Loss: 1.4604 (Cls: 0.3440, Att: 1.1164) | Acc: 82.25% | AUC: 0.9291
Val   — Loss: 0.2423 | Acc: 90.54% | Prec: 0.7807 | Recall: 0.9333 | F1: 0.8502 | AUC: 0.9757
LR: 0.000150

Epoch 6/10
----------------------------------------------------------------------


Val/Test: 100%|██████████| 49/49 [00:13<00:00,  3.54it/s]


Train — Loss: 1.4390 (Cls: 0.3580, Att: 1.0810) | Acc: 82.80% | AUC: 0.9246
Val   — Loss: 0.2763 | Acc: 87.85% | Prec: 0.7138 | Recall: 0.9644 | F1: 0.8204 | AUC: 0.9728
LR: 0.000104

Epoch 7/10
----------------------------------------------------------------------


Train:   3%|▎         | 7/229 [00:12<06:21,  1.72s/it, loss=0.0402, cls=0.0091, att=0.0311, acc=89.29%]


KeyboardInterrupt: 

In [41]:
# ── Load best model and evaluate on test set ──────────────────────────────────
print('\n' + '='*70)
print('Loading best model for test evaluation...')
print('='*70)

checkpoint = torch.load(MODEL_NAME, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"Val F1: {checkpoint['val_f1']:.4f} | Val Recall: {checkpoint['val_recall']:.4f}")

test_loss, test_acc, test_preds, test_labels, test_probs, test_prec, test_recall, test_f1, test_auc = validate(
    model, test_loader, criterion, device
)

print('\n' + '='*70)
print(f'TEST SET RESULTS — Spatial Attention Only (seed={SEED})')
print('='*70)
print(f'Test Loss      : {test_loss:.4f}')
print(f'Test Accuracy  : {test_acc:.2f}%')
print(f'Test AUC-ROC   : {test_auc:.4f}')
print(f'Test Precision : {test_prec:.4f}')
print(f'Test Recall    : {test_recall:.4f}')
print(f'Test F1-Score  : {test_f1:.4f}')

print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=['No Nodule', 'Nodule'], digits=4))

print('Confusion Matrix:')
print(confusion_matrix(test_labels, test_preds))


Loading best model for test evaluation...
Loaded best model from epoch 1
Val F1: 0.8740 | Val Recall: 0.9556


Val/Test: 100%|██████████| 50/50 [00:16<00:00,  3.09it/s]


TEST SET RESULTS — Spatial Attention Only (seed=916)
Test Loss      : 0.2359
Test Accuracy  : 92.87%
Test AUC-ROC   : 0.9837
Test Precision : 0.8084
Test Recall    : 0.9724
Test F1-Score  : 0.8828

Classification Report:
              precision    recall  f1-score   support

   No Nodule     0.9885    0.9120    0.9487       568
      Nodule     0.8084    0.9724    0.8828       217

    accuracy                         0.9287       785
   macro avg     0.8985    0.9422    0.9158       785
weighted avg     0.9388    0.9287    0.9305       785

Confusion Matrix:
[[518  50]
 [  6 211]]


In [ ]:
# ── Small-nodule subset evaluation (fixed) ─────────────────────────────────────
print('\n' + '='*70)
print('Evaluating on small-nodule subset...')
print('='*70)

subset_csv_path = './dataset_nodule21/cxr_images/proccessed_data/subset_metadata2.csv'

# The old path ./.../proccessed_data/images does not exist in your workspace.
candidate_image_dirs = [
    './dataset_nodule21/cxr_images/proccessed_data/split_data/train/images',
    './dataset_nodule21/cxr_images/proccessed_data/split_data/val/images',
    './dataset_nodule21/cxr_images/proccessed_data/split_data/test/images',
]

subset_df = pd.read_csv(subset_csv_path).copy()

def resolve_img_path(img_name):
    for d in candidate_image_dirs:
        p = os.path.join(d, img_name)
        if os.path.exists(p):
            return p
    return None

subset_df['resolved_path'] = subset_df['img_name'].apply(resolve_img_path)
missing = subset_df['resolved_path'].isna().sum()
if missing > 0:
    print(f'Warning: dropping {missing} rows with missing files')

subset_df = subset_df[subset_df['resolved_path'].notna()].reset_index(drop=True)
print(f'Usable rows: {len(subset_df)}')

from PIL import Image
import SimpleITK as sitk
from torch.utils.data import Dataset, DataLoader

class SubsetDatasetResolved(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['resolved_path']

        image = sitk.ReadImage(img_path)
        img_array = sitk.GetArrayFromImage(image)

        if len(img_array.shape) == 3:
            img_array = img_array[0]

        img_array = img_array.astype(np.float32)
        img_min, img_max = img_array.min(), img_array.max()
        if img_max > img_min:
            img_array = ((img_array - img_min) / (img_max - img_min) * 255).astype(np.uint8)
        else:
            img_array = np.zeros_like(img_array, dtype=np.uint8)

        img_array = np.stack([img_array, img_array, img_array], axis=-1)
        image = Image.fromarray(img_array)

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(int(row['label']), dtype=torch.long)
        return image, label

subset_dataset = SubsetDatasetResolved(subset_df, transform=val_transform)
subset_loader = DataLoader(
    subset_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,   # safer on Windows and avoids worker crash masking
    pin_memory=True
)

def validate_no_bbox(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = total = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Subset'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    epoch_loss = running_loss / max(1, len(loader))
    epoch_acc = 100.0 * correct / max(1, total)
    precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0

    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels), all_probs, precision, recall, f1, auc

sub_loss, sub_acc, sub_preds, sub_labels, sub_probs, sub_prec, sub_recall, sub_f1, sub_auc = validate_no_bbox(
    model, subset_loader, criterion, device
)

false_negatives = np.sum((sub_preds == 0) & (sub_labels == 1))
total_positives = np.sum(sub_labels == 1)
correctly_detected = total_positives - false_negatives
detection_rate = correctly_detected / total_positives * 100 if total_positives > 0 else 0.0

print(f'\nSmall-Nodule Subset Results (seed={SEED})')
print(f'  Total nodules      : {total_positives}')
print(f'  Correctly detected : {correctly_detected} ({detection_rate:.1f}%)')
print(f'  False negatives    : {false_negatives} ({100-detection_rate:.1f}%)')
print(f'  Recall             : {sub_recall:.4f}')
print(f'  F1-Score           : {sub_f1:.4f}')
print(f'  AUC                : {sub_auc:.4f}')


Evaluating on small-nodule subset...
Usable rows: 171


Subset: 100%|██████████| 11/11 [00:04<00:00,  2.25it/s]


Small-Nodule Subset Results (seed=444)
  Total nodules      : 171
  Correctly detected : 160 (93.6%)
  False negatives    : 11 (6.4%)
  Recall             : 0.9357
  F1-Score           : 0.9668
  AUC                : 0.0000


In [42]:
# ── Attention-BBox Alignment Metrics ─────────────────────────────────────────
print('\n' + '='*70)
print('Computing Attention-BBox Alignment Metrics...')
print('='*70)

# Need bbox dataset for alignment metrics
subset_bbox_df = pd.read_csv(subset_csv_path).copy()
subset_bbox_df['resolved_path'] = subset_bbox_df['img_name'].apply(resolve_img_path)
subset_bbox_df = subset_bbox_df[subset_bbox_df['resolved_path'].notna()].reset_index(drop=True)

class SubsetDatasetWithBBox(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['resolved_path']

        image = sitk.ReadImage(img_path)
        img_array = sitk.GetArrayFromImage(image)
        if len(img_array.shape) == 3:
            img_array = img_array[0]
        img_array = img_array.astype(np.float32)
        img_min, img_max = img_array.min(), img_array.max()
        if img_max > img_min:
            img_array = ((img_array - img_min) / (img_max - img_min) * 255).astype(np.uint8)
        else:
            img_array = np.zeros_like(img_array, dtype=np.uint8)
        img_array = np.stack([img_array, img_array, img_array], axis=-1)
        image = Image.fromarray(img_array)
        if self.transform:
            image = self.transform(image)

        label = torch.tensor(int(row['label']), dtype=torch.long)

        # Bbox normalized
        img_w = row.get('img_width', 1024)
        img_h = row.get('img_height', 1024)
        if row['label'] == 1:
            bbox = torch.tensor([
                row['x'] / img_w,
                row['y'] / img_h,
                row['width'] / img_w,
                row['height'] / img_h
            ], dtype=torch.float32)
        else:
            bbox = torch.zeros(4, dtype=torch.float32)

        return image, label, bbox

subset_bbox_dataset = SubsetDatasetWithBBox(subset_bbox_df, transform=val_transform)

# Group by unique image — take one row per image
unique_images = subset_bbox_df.drop_duplicates(subset='img_name').reset_index(drop=True)

bbox_coverage_list  = []
detection_rate_list = []
peak_proximity_list = []
attention_focus_list = []

model.eval()
TARGET_SIZE = 224

for idx in tqdm(range(len(unique_images)), desc='Attention Metrics'):
    row = unique_images.iloc[idx]
    img_name = row['img_name']
    label    = int(row['label'])

    if label == 0:
        continue  # skip negatives — no bbox to evaluate against

    # Load image
    img_path  = row['resolved_path']
    image_itk = sitk.ReadImage(img_path)
    arr = sitk.GetArrayFromImage(image_itk)
    if len(arr.shape) == 3:
        arr = arr[0]
    arr = arr.astype(np.float32)
    mn, mx = arr.min(), arr.max()
    if mx > mn:
        arr = ((arr - mn) / (mx - mn) * 255).astype(np.uint8)
    else:
        arr = np.zeros_like(arr, dtype=np.uint8)
    arr = np.stack([arr, arr, arr], axis=-1)
    pil_img = Image.fromarray(arr)
    img_tensor = val_transform(pil_img).unsqueeze(0).to(device)

    # Get attention map
    with torch.no_grad():
        attention_map = model.get_attention_map(img_tensor)
        attention_map = attention_map.squeeze().cpu().numpy()

    # Resize to 224x224
    attention_map = cv2.resize(attention_map, (TARGET_SIZE, TARGET_SIZE))

    # Normalize attention to [0, 1]
    a_min, a_max = attention_map.min(), attention_map.max()
    if a_max > a_min:
        attention_map = (attention_map - a_min) / (a_max - a_min)

    # Get ALL bboxes for this image
    img_rows = subset_bbox_df[subset_bbox_df['img_name'] == img_name]
    img_w = img_rows.iloc[0].get('img_width', 1024)
    img_h = img_rows.iloc[0].get('img_height', 1024)

    img_bbox_coverage  = []
    img_detection_rate = []
    img_peak_proximity = []
    img_attention_focus = []

    for _, brow in img_rows.iterrows():
        if brow['label'] != 1:
            continue

        x = (brow['x'] / img_w) * TARGET_SIZE
        y = (brow['y'] / img_h) * TARGET_SIZE
        w = (brow['width'] / img_w) * TARGET_SIZE
        h = (brow['height'] / img_h) * TARGET_SIZE

        x_pix = max(0, int(x))
        y_pix = max(0, int(y))
        x_end = min(TARGET_SIZE, int(x + w))
        y_end = min(TARGET_SIZE, int(y + h))

        # BBox mask
        bbox_mask = np.zeros((TARGET_SIZE, TARGET_SIZE), dtype=np.float32)
        bbox_mask[y_pix:y_end, x_pix:x_end] = 1.0
        bbox_area = bbox_mask.sum()

        if bbox_area == 0:
            continue

        # 1. BBox Coverage — mean attention inside bbox
        coverage = (attention_map * bbox_mask).sum() / bbox_area
        img_bbox_coverage.append(float(coverage))

        # 2. Detection Rate — >20% of bbox in high-attention region (threshold 0.5)
        high_att = (attention_map > 0.5).astype(np.float32)
        overlap_ratio = (high_att * bbox_mask).sum() / bbox_area
        img_detection_rate.append(1.0 if overlap_ratio > 0.2 else 0.0)

        # 3. Peak Proximity — normalized distance from attention peak to bbox center
        peak_y, peak_x = np.unravel_index(np.argmax(attention_map), attention_map.shape)
        bbox_cx = x + w / 2
        bbox_cy = y + h / 2
        dist = np.sqrt((peak_x - bbox_cx) ** 2 + (peak_y - bbox_cy) ** 2)
        max_dist = np.sqrt(TARGET_SIZE ** 2 + TARGET_SIZE ** 2)
        proximity = max(0.0, 1.0 - dist / max_dist)
        img_peak_proximity.append(float(proximity))

        # 4. Attention Focus — ratio of mean attention inside vs outside bbox
        outside_mask = 1.0 - bbox_mask
        mean_inside  = (attention_map * bbox_mask).sum() / (bbox_area + 1e-8)
        mean_outside = (attention_map * outside_mask).sum() / (outside_mask.sum() + 1e-8)
        focus = min(mean_inside / (mean_outside + 1e-8), 10.0)
        img_attention_focus.append(float(focus))

    if img_bbox_coverage:
        bbox_coverage_list.append(np.mean(img_bbox_coverage))
        detection_rate_list.append(np.mean(img_detection_rate))
        peak_proximity_list.append(np.max(img_peak_proximity))
        attention_focus_list.append(np.mean(img_attention_focus))

print(f'\nAttention-BBox Alignment Metrics (seed={SEED})')
print(f'  BBox Coverage  : {np.mean(bbox_coverage_list):.4f}')
print(f'  Detection Rate : {np.mean(detection_rate_list):.4f}  ({np.mean(detection_rate_list)*100:.2f}%)')
print(f'  Peak Proximity : {np.mean(peak_proximity_list):.4f}')
print(f'  Attention Focus: {np.mean(attention_focus_list):.4f}')


Computing Attention-BBox Alignment Metrics...


Attention Metrics: 100%|██████████| 135/135 [00:04<00:00, 28.39it/s]


Attention-BBox Alignment Metrics (seed=916)
  BBox Coverage  : 0.4915
  Detection Rate : 0.6012  (60.12%)
  Peak Proximity : 0.7379
  Attention Focus: 2.5314
